# Preparação dos Dados — do OHLCV bruto ao tensor de treino (CNN+LSTM)

**Aula 2 · Predição de operação em um único ativo:** amanhã compro, vendo ou mantenho?

Este notebook implementa, etapa por etapa, o pipeline apresentado nos slides:

| # | Etapa | Onde o futuro pode vazar |
|---|-------|--------------------------|
| 1 | Dados brutos (OHLCV ajustado) | dados não ajustados por splits |
| 2 | Features (retornos, vol, indicadores) | janelas centradas, `shift(-1)` |
| 3 | Rotulagem (threshold / triple barrier) | **aqui o futuro entra de propósito, só no alvo** |
| 4 | Split temporal + purging + embargo | rótulo do treino sobrepondo o teste |
| 5 | Normalização (scaler só no treino) | `scaler.fit(série inteira)` |
| 6 | Janelamento (T, F) | janela cruzando a fronteira treino/teste |
| 7 | Tensores X:(N,T,F), y:(N,3) + testes de sanidade | — |

> **Ordem importa:** features → rótulos → split → scaler → janelas. Nunca o contrário.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
np.random.seed(42)

# ---- Hiperparâmetros da preparação (fixe-os por critério econômico, não olhando o teste) ----
H       = 5      # horizonte do rótulo, em dias
K       = 1.0    # multiplicador da volatilidade no threshold θ
T       = 30     # dias por janela
EMBARGO = 5      # dias extras de lacuna entre blocos
CLIP_Z  = 5.0    # recorte de outliers após z-score

## 1 · Dados brutos: OHLCV diário

Use um único ativo líquido com 8–10 anos de histórico e **preços ajustados** por splits e dividendos.

A célula abaixo tenta baixar dados reais com `yfinance`. Se a biblioteca ou a rede não estiverem disponíveis, gera uma série **sintética** com regimes de volatilidade diferentes — suficiente para exercitar todo o pipeline (mas sem nenhum sinal real de mercado).

In [ ]:
def load_real(ticker="PETR4.SA", start="2016-01-01"):
    import yfinance as yf
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]
    df.index.name = "date"
    return df.dropna()

def load_synthetic(n_years=10, seed=42):
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range("2016-01-04", periods=252 * n_years)
    n = len(idx)
    # regimes de volatilidade que se alternam (calmo / normal / crise)
    regime = np.repeat(rng.choice([0.008, 0.014, 0.03], size=n // 120 + 1, p=[0.45, 0.4, 0.15]), 120)[:n]
    ret = rng.normal(0.0003, 1.0, n) * regime
    close = 30 * np.exp(np.cumsum(ret))
    open_ = close * np.exp(rng.normal(0, 0.003, n))
    hi = np.maximum(open_, close) * np.exp(np.abs(rng.normal(0, 0.006, n)))
    lo = np.minimum(open_, close) * np.exp(-np.abs(rng.normal(0, 0.006, n)))
    vol = (rng.lognormal(15, 0.4, n) * (1 + 8 * regime)).round()
    return pd.DataFrame({"open": open_, "high": hi, "low": lo, "close": close, "volume": vol}, index=idx).rename_axis("date")

try:
    raw = load_real()
    SOURCE = "yfinance"
except Exception as e:
    raw = load_synthetic()
    SOURCE = f"sintético ({type(e).__name__})"

raw_backup = raw.copy()          # cópia crua, antes de qualquer transformação
print("Fonte:", SOURCE, "| linhas:", len(raw), "| período:", raw.index.min().date(), "→", raw.index.max().date())
raw.head()

### Verificações obrigatórias

Datas duplicadas/faltantes, valores não positivos, coerência de High/Low e saltos absurdos (split não ajustado ou erro da fonte).

In [ ]:
def check_ohlcv(df, jump_thr=0.30):
    problems = {}
    problems["datas_duplicadas"] = int(df.index.duplicated().sum())
    problems["nao_monotonico"]   = not df.index.is_monotonic_increasing
    gaps = df.index.to_series().diff().dt.days
    problems["buracos_>5_dias"]  = int((gaps > 5).sum())          # feriados longos são raros; buracos maiores merecem olhar
    problems["preco_nao_positivo"] = int((df[["open", "high", "low", "close"]] <= 0).any(axis=1).sum())
    problems["volume_nao_positivo"] = int((df["volume"] <= 0).sum())
    problems["high_incoerente"] = int((df["high"] < df[["open", "close"]].max(axis=1)).sum())
    problems["low_incoerente"]  = int((df["low"]  > df[["open", "close"]].min(axis=1)).sum())
    problems["saltos_>30%"]     = int((df["close"].pct_change().abs() > jump_thr).sum())
    return pd.Series(problems, name="ocorrências")

check_ohlcv(raw)

## Por que não alimentar a rede com preço

Preço é não-estacionário; o log-retorno $r_t = \ln(P_t / P_{t-1})$ oscila em torno de zero, tem escala comparável ao longo dos anos e é aditivo no tempo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
raw["close"].plot(ax=ax[0], color="#0F1B2D", title="Preço ajustado")
np.log(raw["close"]).diff().plot(ax=ax[1], color="#3B7DD8", lw=0.6, title="Log-retorno diário")
for a in ax: a.set_xlabel("")
plt.tight_layout(); plt.show()

## 2 · Features

Três regras:

1. **Só passado** — a linha $t$ usa dados de $t, t-1, \dots$; nunca $t+1$. Todo `rolling` termina em $t$.
2. **Sem níveis absolutos** — tudo é retorno, razão ou diferença normalizada. `SMA20` é nível; `C/SMA20 − 1` é feature.
3. **Descarte o warm-up** — as primeiras linhas com `NaN` são removidas, não preenchidas com zero.

Como cada linha só olha para trás, é permitido calcular as features sobre a série inteira **antes** do split.

In [ ]:
def rsi(close, n=14):
    d = close.diff()
    up, dn = d.clip(lower=0), -d.clip(upper=0)
    rs = up.ewm(alpha=1 / n, adjust=False).mean() / dn.ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + rs)

def build_features(df):
    o, h, l, c, v = df["open"], df["high"], df["low"], df["close"], df["volume"]
    f = pd.DataFrame(index=df.index)

    # --- retornos e volatilidade ---
    f["ret_1"] = np.log(c / c.shift(1))
    for k in (5, 10, 20):
        f[f"ret_{k}"] = np.log(c / c.shift(k))
    f["vol_10"] = f["ret_1"].rolling(10).std()
    f["vol_20"] = f["ret_1"].rolling(20).std()
    f["vol_ratio"] = f["vol_10"] / f["vol_20"]
    f["range"] = (h - l) / c
    f["gap"] = (o - c.shift(1)) / c.shift(1)
    f["body"] = (c - o) / (h - l).replace(0, np.nan)
    f["ret_1_z"] = (f["ret_1"] - f["ret_1"].rolling(20).mean()) / f["vol_20"]

    # --- tendência ---
    sma20, sma50 = c.rolling(20).mean(), c.rolling(50).mean()
    f["dist_sma20"] = c / sma20 - 1
    f["dist_sma50"] = c / sma50 - 1
    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    f["macd_hist"] = (macd - macd.ewm(span=9, adjust=False).mean()) / c
    f["sma20_slope"] = sma20 / sma20.shift(5) - 1

    # --- osciladores ---
    f["rsi"] = rsi(c, 14) / 50 - 1                                   # [-1, 1]
    bb_mid, bb_sd = c.rolling(20).mean(), c.rolling(20).std()
    f["pct_b"] = (c - (bb_mid - 2 * bb_sd)) / (4 * bb_sd)
    f["bb_width"] = 4 * bb_sd / c
    lo14, hi14 = l.rolling(14).min(), h.rolling(14).max()
    f["stoch_k"] = (c - lo14) / (hi14 - lo14).replace(0, np.nan) * 2 - 1

    # --- volume ---
    f["vol_rel"] = v / v.rolling(20).mean()
    f["dlog_vol"] = np.log(v / v.shift(1))

    # --- calendário ---
    dow = df.index.dayofweek
    f["dow_sin"] = np.sin(2 * np.pi * dow / 5)
    f["dow_cos"] = np.cos(2 * np.pi * dow / 5)
    return f

features = build_features(raw)
print("features:", list(features.columns))
print("linhas com NaN (warm-up):", features.isna().any(axis=1).sum())

### Teste da série truncada (regra 1)

Recalcule as features com a série cortada em um dia $t$ qualquer. A linha $t$ tem que ser idêntica à da série completa. Se não for, alguma feature está olhando para frente.

In [ ]:
def truncation_test(df, builder, n_checks=5, seed=0):
    rng = np.random.default_rng(seed)
    full = builder(df)
    for t in rng.choice(np.arange(100, len(df)), n_checks, replace=False):
        part = builder(df.iloc[: t + 1])
        assert np.allclose(full.iloc[t].values, part.iloc[t].values, equal_nan=True), f"vazamento na linha {df.index[t].date()}"
    return "OK: nenhuma feature usa o futuro"

truncation_test(raw, build_features)

### Correlação entre features

Pares com |ρ| > 0,95 dizem quase a mesma coisa; remova um deles.

In [ ]:
corr = features.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, bool), k=1))
redundant = [(a, b, round(float(upper.loc[a, b]), 3)) for a in upper.index for b in upper.columns if upper.loc[a, b] > 0.95]
print("pares redundantes:", redundant or "nenhum")

## 3 · Rotulagem

Aqui o futuro entra **de propósito** — apenas no alvo. Duas opções:

**(a) Threshold sobre retorno futuro:** $R_t = \ln(C_{t+h}/C_t)$, comparado com $\theta_t = k \cdot \text{vol}_{20}(t) \cdot \sqrt{h}$.
θ proporcional à volatilidade evita que anos calmos virem só "mantém" e anos de crise virem só compra/venda.

**(b) Triple Barrier (López de Prado):** o rótulo é a primeira barreira tocada — alvo $C_t(1+\theta)$, stop $C_t(1-\theta)$ ou o limite de tempo $t+h$. Usa High/Low para checar o toque.

Codificação: `0 = venda`, `1 = mantém`, `2 = compra`. As últimas $h$ linhas não têm futuro conhecido e são descartadas.

In [ ]:
CLASSES = {0: "venda", 1: "mantém", 2: "compra"}

def theta_series(features, k=K, h=H):
    return k * features["vol_20"] * np.sqrt(h)

def label_threshold(close, theta, h=H):
    fwd = np.log(close.shift(-h) / close)
    y = pd.Series(1, index=close.index, dtype="float")
    y[fwd > theta] = 2
    y[fwd < -theta] = 0
    y[fwd.isna() | theta.isna()] = np.nan
    return y

def label_triple_barrier(df, theta, h=H, tie_break_sign=False):
    c, hi, lo = df["close"].values, df["high"].values, df["low"].values
    th = theta.values
    n = len(c)
    y = np.full(n, np.nan)
    for t in range(n - h):
        if np.isnan(th[t]):
            continue
        up, dn = c[t] * (1 + th[t]), c[t] * (1 - th[t])
        label = 1
        for j in range(t + 1, t + h + 1):
            hit_up, hit_dn = hi[j] >= up, lo[j] <= dn
            if hit_up and hit_dn:          # ambos no mesmo dia: assume o pior caso (stop)
                label = 0; break
            if hit_up:
                label = 2; break
            if hit_dn:
                label = 0; break
        if label == 1 and tie_break_sign:  # variante: sinal do retorno final
            label = 2 if c[t + h] > c[t] else 0
        y[t] = label
    return pd.Series(y, index=df.index)

theta = theta_series(features)
y_thr = label_threshold(raw["close"], theta)
y_tb  = label_triple_barrier(raw, theta)

dist = pd.DataFrame({
    "threshold": y_thr.value_counts(normalize=True).sort_index(),
    "triple_barrier": y_tb.value_counts(normalize=True).sort_index(),
}).rename(index=CLASSES).round(3)
dist

### Classes desbalanceadas

"Mantém" domina. Responder "mantém" para tudo já dá a acurácia da classe majoritária — sem aprender nada. Por isso:

- **pesos na loss**: $w_c = N / (3 \cdot N_c)$, calculados **só no treino** (feito na seção 4);
- métrica correta desde já: **F1 por classe** e matriz de confusão, nunca acurácia global;
- nada de SMOTE em sequências temporais.

Escolha aqui qual rotulagem seguir para o resto do notebook.

In [ ]:
LABELING = "triple_barrier"          # ou "threshold"
y_all = (y_tb if LABELING == "triple_barrier" else y_thr).rename("y")

ax = dist.plot.bar(figsize=(6, 3), color=["#3B7DD8", "#19A974"], rot=0, title="Distribuição das classes")
ax.set_ylabel("fração"); plt.tight_layout(); plt.show()

### Montagem da tabela final e descarte do warm-up e das últimas h linhas

In [ ]:
data = features.join(y_all)
n_before = len(data)
data = data.dropna()                        # remove warm-up (início) e linhas sem rótulo (fim)
FEATURES = list(features.columns)
print(f"{n_before} → {len(data)} linhas | {len(FEATURES)} features | {data.index.min().date()} → {data.index.max().date()}")
data.tail(3)

## 4 · Split temporal, purging e embargo

Nunca embaralhe. O teste é o futuro do treino, com uma lacuna entre eles:

- **Purging** — o rótulo do dia $t$ usa $C_{t+h}$; as últimas $h$ linhas do treino "sabem" o início do teste. Descarte-as.
- **Embargo** — alguns dias extras de lacuna protegem contra autocorrelação residual (janelas vizinhas são quase idênticas).

Primeiro um split simples (treino / validação / teste), depois o gerador de janelas **walk-forward**.

In [ ]:
@dataclass
class Split:
    train: pd.DataFrame
    val: pd.DataFrame
    test: pd.DataFrame

def temporal_split(data, val_start, test_start, h=H, embargo=EMBARGO):
    gap = h + embargo
    train = data.loc[: val_start].iloc[:-gap]                      # purging + embargo antes da validação
    val   = data.loc[val_start: test_start].iloc[:-gap]            # idem antes do teste
    test  = data.loc[test_start:]
    return Split(train, val, test)

def describe_split(sp):
    rows = []
    for name, blk in (("treino", sp.train), ("validação", sp.val), ("teste", sp.test)):
        d = blk["y"].value_counts(normalize=True).reindex([0, 1, 2]).fillna(0)
        rows.append({"bloco": name, "início": blk.index.min().date(), "fim": blk.index.max().date(), "n": len(blk),
                     "venda": round(d[0], 3), "mantém": round(d[1], 3), "compra": round(d[2], 3)})
    return pd.DataFrame(rows).set_index("bloco")

last_year = data.index.max().year
sp = temporal_split(data, val_start=f"{last_year - 1}-01-01", test_start=f"{last_year}-01-01")
describe_split(sp)

**Datas nas pontas:** confirme com os olhos que `treino < lacuna < validação < lacuna < teste`. A célula acima já mostra a distribuição por bloco; diferenças grandes indicam mudança de regime, não erro — mas você precisa saber disso antes de treinar.

In [ ]:
def walk_forward(data, first_test_year, h=H, embargo=EMBARGO, val_months=12):
    '''Gera (treino, validação, teste) ano a ano. Cada bloco de teste é fora da amostra de verdade.'''
    gap = h + embargo
    for year in range(first_test_year, data.index.max().year + 1):
        test_start = pd.Timestamp(f"{year}-01-01")
        val_start = test_start - pd.DateOffset(months=val_months)
        train = data.loc[: val_start].iloc[:-gap]
        val = data.loc[val_start: test_start].iloc[:-gap]
        test = data.loc[test_start: pd.Timestamp(f"{year}-12-31")]
        if len(train) < 500 or len(test) == 0:
            continue
        yield year, Split(train, val, test)

for year, s in walk_forward(data, first_test_year=data.index.min().year + 4):
    print(f"testa {year}: treino {s.train.index.min().date()}→{s.train.index.max().date()} ({len(s.train)}) | "
          f"val →{s.val.index.max().date()} ({len(s.val)}) | teste {s.test.index.min().date()}→{s.test.index.max().date()} ({len(s.test)})")

### Pesos de classe — calculados apenas no treino

In [ ]:
def class_weights(y):
    counts = y.value_counts().reindex([0, 1, 2]).fillna(0).values
    return {c: float(w) for c, w in zip([0, 1, 2], len(y) / (3 * counts))}

cw = class_weights(sp.train["y"])
{CLASSES[k]: round(v, 3) for k, v in cw.items()}

## 5 · Normalização: ajustar no treino, aplicar no resto

O scaler é um modelo. Se ele vê o teste, vaza. Média e desvio vêm **só do treino**, e cada janela do walk-forward recebe um scaler novo.

- Z-score por feature, com recorte de outliers em ±5σ (um dia de −15% não deve dominar a escala).
- Features já limitadas (RSI, %B, seno/cosseno) podem ficar sem normalizar — aqui, por simplicidade, normalizamos tudo exceto o calendário.

In [ ]:
NO_SCALE = ["dow_sin", "dow_cos"]

class TrainOnlyScaler:
    def __init__(self, clip=CLIP_Z, skip=NO_SCALE):
        self.clip, self.skip = clip, skip
    def fit(self, X: pd.DataFrame):
        cols = [c for c in X.columns if c not in self.skip]
        self.cols = cols
        self.mu = X[cols].mean()
        self.sd = X[cols].std().replace(0, 1.0)
        return self
    def transform(self, X: pd.DataFrame):
        Z = X.copy()
        Z[self.cols] = ((X[self.cols] - self.mu) / self.sd).clip(-self.clip, self.clip)
        return Z

scaler = TrainOnlyScaler().fit(sp.train[FEATURES])       # <-- só o treino entra aqui
train_s, val_s, test_s = (scaler.transform(b[FEATURES]).join(b["y"]) for b in (sp.train, sp.val, sp.test))

pd.DataFrame({"treino_média": train_s[FEATURES].mean(), "treino_sd": train_s[FEATURES].std(),
              "teste_média": test_s[FEATURES].mean(), "teste_sd": test_s[FEATURES].std()}).round(2).head(8)

## 6 · Janelamento: da tabela às sequências (T, F)

Cada amostra é uma "foto" dos últimos $T$ dias. A janela termina em $t$ (a feature de $t$ está incluída, a de $t+1$ não); o rótulo é a decisão para $t+1$, calculada com $C_{t+1+h}$.

Janelar **depois** do split garante que nenhuma janela cruza a fronteira treino/teste. Stride = 1.

In [ ]:
def make_windows(block: pd.DataFrame, T=T, features=FEATURES):
    '''Retorna X:(N,T,F), y:(N,), datas:(N,) — a data é o dia t+1 (dia da decisão).'''
    Xf = block[features].values.astype("float32")
    yv = block["y"].values.astype("int64")
    N = len(block) - T
    if N <= 0:
        return np.empty((0, T, len(features)), "float32"), np.empty((0,), "int64"), block.index[:0]
    idx = np.arange(T)[None, :] + np.arange(N)[:, None]          # (N, T) índices deslizantes
    X = Xf[idx]                                                   # janela: linhas t-T+1 … t
    y = yv[T:]                                                    # rótulo: decisão em t+1
    dates = block.index[T:]
    return X, y, dates

X_tr, y_tr, d_tr = make_windows(train_s)
X_va, y_va, d_va = make_windows(val_s)
X_te, y_te, d_te = make_windows(test_s)

def to_onehot(y, n=3):
    return np.eye(n, dtype="float32")[y]

for name, X, y in (("treino", X_tr, y_tr), ("validação", X_va, y_va), ("teste", X_te, y_te)):
    print(f"{name:10s} X {X.shape}  y {y.shape}  one-hot {to_onehot(y).shape}")

In [ ]:
# Conferência visual de uma janela: a última linha da janela é o dia t; o rótulo refere-se a t+1.
i = 0
print("janela cobre:", train_s.index[i].date(), "→", train_s.index[i + T - 1].date())
print("decisão para:", d_tr[i].date(), "| rótulo:", CLASSES[int(y_tr[i])])
assert np.allclose(X_tr[i, -1], train_s[FEATURES].iloc[i + T - 1].values)

## 7 · Antes de treinar: testes de sanidade

Cinco checagens baratas que pegam a maioria dos vazamentos. Usamos uma regressão logística sobre a **última linha** de cada janela como modelo barato:

1. **Baseline ingênuo** — o piso que o CNN+LSTM tem que superar.
2. **Teste do embaralhamento** — com rótulos embaralhados, o F1 macro deve cair para ~acaso. Se ficar alto, há vazamento.
3. **Teste da série truncada** — já feito na seção 2.
4. **Distribuição por bloco** — já vista na seção 4.
5. **Datas nas pontas** — já vistas na seção 4.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def last_row(X):           # (N, T, F) -> (N, F)
    return X[:, -1, :]

def fit_logit(X, y, weights):
    clf = LogisticRegression(max_iter=2000, class_weight=weights)
    return clf.fit(last_row(X), y)

baseline = fit_logit(X_tr, y_tr, cw)
pred_va = baseline.predict(last_row(X_va))
majority = np.full_like(y_va, np.bincount(y_tr).argmax())

print("F1 macro — baseline logístico (validação):", round(f1_score(y_va, pred_va, average="macro"), 3))
print("F1 macro — classe majoritária:            ", round(f1_score(y_va, majority, average="macro"), 3))
print("\nmatriz de confusão (linhas = real, colunas = previsto; 0=venda 1=mantém 2=compra)")
print(confusion_matrix(y_va, pred_va))
print(classification_report(y_va, pred_va, target_names=[CLASSES[i] for i in range(3)], zero_division=0))

In [ ]:
rng = np.random.default_rng(0)
scores = []
for _ in range(5):
    y_shuf = rng.permutation(y_tr)
    clf = fit_logit(X_tr, y_shuf, class_weights(pd.Series(y_shuf)))
    scores.append(f1_score(y_va, clf.predict(last_row(X_va)), average="macro"))
print("F1 macro com rótulos embaralhados (5 rodadas):", np.round(scores, 3))
print("→ deve ficar no nível do acaso (~0.2–0.35 para 3 classes). Se ficar acima do baseline real, há vazamento.")

## Pipeline completo por janela do walk-forward

Tudo junto, na ordem certa, para cada bloco de teste: **split → scaler (só treino) → janelas → pesos de classe**. É esta função que a aula 3 vai chamar para treinar o CNN+LSTM.

In [ ]:
def prepare_fold(sp: Split, T=T, features=FEATURES):
    scaler = TrainOnlyScaler().fit(sp.train[features])
    out = {}
    for name, blk in (("train", sp.train), ("val", sp.val), ("test", sp.test)):
        X, y, dates = make_windows(scaler.transform(blk[features]).join(blk["y"]), T, features)
        out[f"X_{name}"], out[f"y_{name}"], out[f"dates_{name}"] = X, y, dates
    out["class_weights"] = class_weights(sp.train["y"])
    out["scaler"] = scaler
    return out

folds = {}
for year, s in walk_forward(data, first_test_year=data.index.min().year + 4):
    f = prepare_fold(s)
    folds[year] = f
    # sanidade: janelas não cruzam fronteiras e há lacuna entre blocos
    assert f["dates_train"].max() < s.val.index.min() and f["dates_val"].max() < s.test.index.min()
    print(f"{year}: X_train {f['X_train'].shape} | X_val {f['X_val'].shape} | X_test {f['X_test'].shape} | "
          f"pesos {dict((CLASSES[k], round(v, 2)) for k, v in f['class_weights'].items())}")

### Baseline logístico em walk-forward

Registre o piso para cada ano. Um modelo que brilha em um ano e afunda no seguinte não aprendeu nada estrutural.

In [ ]:
rows = []
for year, f in folds.items():
    clf = fit_logit(f["X_train"], f["y_train"], f["class_weights"])
    p = clf.predict(last_row(f["X_test"]))
    rows.append({"ano_teste": year, "n_teste": len(p),
                 "f1_macro": f1_score(f["y_test"], p, average="macro"),
                 "f1_compra": f1_score(f["y_test"], p, labels=[2], average="macro", zero_division=0),
                 "f1_venda": f1_score(f["y_test"], p, labels=[0], average="macro", zero_division=0)})
pd.DataFrame(rows).set_index("ano_teste").round(3)

## Salvando os tensores

Um arquivo `.npz` por janela do walk-forward, com X, y (índices e one-hot) e as datas de decisão. A aula 3 começa daqui.

In [ ]:
import os
os.makedirs("tensores", exist_ok=True)
for year, f in folds.items():
    np.savez_compressed(
        f"tensores/fold_{year}.npz",
        X_train=f["X_train"], y_train=f["y_train"], y_train_onehot=to_onehot(f["y_train"]),
        X_val=f["X_val"], y_val=f["y_val"], y_val_onehot=to_onehot(f["y_val"]),
        X_test=f["X_test"], y_test=f["y_test"], y_test_onehot=to_onehot(f["y_test"]),
        dates_test=f["dates_test"].values.astype("datetime64[D]"),
        class_weights=np.array([f["class_weights"][k] for k in range(3)]),
        features=np.array(FEATURES),
    )
sorted(os.listdir("tensores"))

## Checklist da preparação

- [x] OHLCV ajustado, sem buracos nem saltos absurdos (`check_ohlcv`)
- [x] Todas as features são retornos, razões ou diferenças; nenhuma em unidade de preço
- [x] Cada feature usa apenas dados até $t$ (`truncation_test`); warm-up descartado, não preenchido
- [x] Rótulo com θ proporcional à volatilidade (threshold ou triple barrier)
- [x] Últimas $h$ linhas sem rótulo removidas; pesos de classe calculados no treino
- [x] Split temporal com purging ≥ $h$ dias e embargo entre janelas do walk-forward
- [x] Scaler ajustado só no treino; um scaler por janela (`prepare_fold`)
- [x] Janelas (T, F) montadas depois do split, sem cruzar fronteiras (assert)
- [x] Teste de embaralhamento passou; baseline logístico registrado por ano

**Próxima aula:** Conv1D → LSTM → softmax, treino com walk-forward e backtest com custos.